In [1]:
import pandas as pd
import numpy as np
import gensim
import re
import os
import nltk
from tqdm import tqdm 
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from gensim.similarities import WmdSimilarity
from gensim.models import Phrases

pd.set_option('display.max_colwidth', None)

In [2]:
jobs_data_dir = "Job Data\\"
dataframes = []
# Read all files in the directory
for file_name in os.listdir(jobs_data_dir):
    if '2024' in file_name: 
        print(file_name)
        file_path = os.path.join(jobs_data_dir, file_name)
        tmp_df = pd.read_csv(file_path, usecols = ['jobPostingId', 'title', 'description', 'tokens', 'gender_category'])
        tmp_df = tmp_df.dropna(how='any')
        dataframes.append(tmp_df)
job_postings_df = pd.concat(dataframes)
print(len(job_postings_df))
job_postings_df.head()

annotated_data_2024_batch_2_removed_words_tokenized.csv
annotated_data_2024_removed_words_tokenized.csv
77975


,jobPostingId,title,description,gender_category,tokens
0,3823301365,Python Full Stack Developer/Lead/Architect,"Skills- Python, Django, React/AngularExperience- 4+ yrsLocation- Hyderabad\nJob Description-\nâ¢ Python + Django, Angular/React Develop, and maintain robust, scalable backend systems using Python.â¢ Ensure technology solutions align with company architectural standards.â¢ Collaborate seamlessly with cross-functional teams for the integration of platform services.â¢ Foster close collaboration with frontend developers, ensuring a harmonious backend-frontend integration.â¢ Proactively identify and resolve issues promptly.â¢ Strong full-stack development skills in Python, including expertise in frameworks like Django (Expert).â¢ Experience with microservice architecture, end-to-end UI/API integration, and knowledge of API protocols like REST, gRPC, and GraphQL (Advanced).â¢ Knowledge of Caching technologies and DBMS technologies like MySQL, PostGres, MongoDB, and database schema design (Advanced).â¢ Strong problem-solving, communication, and organizational skills (Advanced).â¢ Proficiency in UI modern x like AngularJS or ReactJS (Intermediate).â¢ Proficient in drafting coding practices and designing highly scalable, secure, and maintainable software solutions (Intermediate).â¢ Experience in building large-scale platforms handling real-time complex transactions and troubleshooting distributed systems (Intermediate).",fem,"['python', 'full', 'stack', 'python', 'django', 'hyderabad', 'job', 'python', 'django', 'develop', 'maintain', 'robust', 'scalable', 'backend', 'system', 'using', 'ensure', 'technology', 'solution', 'align', 'company', 'architectural', 'collaborate', 'seamlessly', 'team', 'integration', 'platform', 'foster', 'close', 'collaboration', 'frontend', 'developer', 'ensuring', 'harmonious', 'proactively', 'identify', 'resolve', 'issue', 'strong', 'development', 'skill', 'python', 'including', 'expertise', 'framework', 'like', 'django', 'expert', 'experience', 'microservice', 'architecture', 'integration', 'knowledge', 'api', 'protocol', 'like', 'rest', 'grpc', 'graphql', 'advanced', 'knowledge', 'caching', 'technology', 'dbms', 'technology', 'like', 'mysql', 'postgres', 'mongodb', 'database', 'schema', 'design', 'advanced', 'strong', 'communication', 'organizational', 'skill', 'advanced', 'proficiency', 'ui', 'modern', 'x', 'like', 'angularjs', 'reactjs', 'intermediate', 'proficient', 'drafting', 'coding', 'practice', 'designing', 'highly', 'scalable', 'secure', 'maintainable', 'software', 'solution', 'intermediate', 'experience', 'building', 'platform', 'handling', 'complex', 'transaction', 'troubleshooting', 'distributed', 'system', 'intermediate']"
1,3823301077,Senior Data Engineer - w2 only,"Expertise:8+ years of relevant industry experience with a BS/MastersExperience with distributed processing technologies and frameworks, such as Hadoop, Spark, Kafka, and distributed storage systems (e.g., HDFS, S3)Demonstrated ability to analyze large data sets to identify gaps and inconsistencies, provide data insights, and advance effective product solutionsExpertise with ETL schedulers such as Apache Airflow, Luigi, Oozie, AWS Glue or similar frameworksSolid understanding of data warehousing concepts and hands-on experience with relational databases (e.g., PostgreSQL, MySQL) and columnar databases (e.g., Redshift, BigQuery, HBase, ClickHouse)Excellent written and verbal communication skillsA Typical Day:Design, build, and maintain robust and efficient data pipelines that collect, process, and store data from various sources, including user interactions, financial details, and external data feeds.Develop data models that enable the efficient analysis and manipulation of data for merchandising optimization. Ensure data quality, consistency, and accuracy.Build scalable data pipelines (SparkSQL & Scala) leveraging Airflow scheduler/executor frameworkCollaborate with cross-functional teams, including Data Scie

In [3]:
cvs_df = pd.read_csv("User Data/cvs_tokenized_final.csv")
cvs_df.head()

,CV,gender,text,tokens
0,"```plaintext\n**Summary** \nDynamic and detail-oriented software engineer with 12-14 years of progressive experience in developing robust applications using a variety of programming languages including TypeScript, SQL, and Java. Proven expertise in leveraging web and database technologies such as Node.js, ASP.NET Core, Firebase, and MongoDB. Highly adept in employing modern frameworks and tools to streamline development processes and enhance application performance. Committed to delivering innovative solutions and continuous improvement in the software development lifecycle.\n\n**Skills** \n- **Programming Languages:** TypeScript, SQL, Java \n- **Databases:** Firebase, MongoDB \n- **Web Frameworks:** Node.js, ASP.NET Core \n- **Other Frameworks:** .NET, Keras \n- **Tools:** Git, Ansible \n- **IDEs:** Sublime Text, IntelliJ \n- **Operating Systems:** Windows, macOS \n\n**Experience** \n- Developed and maintained complex software applications, leading projects from conception through deployment. \n- Collaborated with cross-functional teams to design and implement scalable solutions, enhancing user experience and operational efficiency. \n- Utilized TypeScript and SQL for backend services and database management, demonstrating fluency in data-driven application development. \n- Leveraged frameworks such as Node.js and ASP.NET Core to build responsive web applications, driving increased engagement and usability. \n- Implemented CI/CD practices using Git and Ansible, resulting in improved deployment times and reduced rollout errors. \n\n**Education** \nBachelorâ€™s Degree in Computer Science \n[Your University Name] \n[Year of Graduation] \nBrazil \n```",female,"```plaintext\n**Summary** \nDynamic and detail-oriented software engineer with 12-14 years of progressive experience in developing robust applications using a variety of programming languages including TypeScript, SQL, and Java. Proven expertise in leveraging web and database technologies such as Node.js, ASP.NET Core, Firebase, and MongoDB. Highly adept in employing modern frameworks and tools to streamline development processes and enhance application performance. Committed to delivering innovative solutions and continuous improvement in the software development lifecycle.\n\n**Skills** \n- **Programming Languages:** TypeScript, SQL, Java \n- **Databases:** Firebase, MongoDB \n- **Web Frameworks:** Node.js, ASP.NET Core \n- **Other Frameworks:** .NET, Keras \n- **Tools:** Git, Ansible \n- **IDEs:** Sublime Text, IntelliJ \n- **Operating Systems:** Windows, macOS \n\n**Experience** \n- Developed and maintained complex software applications, leading projects from conception through deployment. \n- Collaborated with cross-functional teams to design and implement scalable solutions, enhancing user experience and operational efficiency. \n- Utilized TypeScript and SQL for backend services and database management, demonstrating fluency in data-driven application development. \n- Leveraged frameworks such as Node.js and ASP.NET Core to build responsive web applications, driving increased engagement and usability. \n- Implemented CI/CD practices using Git and Ansible, resulting in improved deployment times and reduced rollout errors. \n\n**Education** \nBachelorâ€™s Degree in Computer Science \n[Your University Name] \n[Year of Graduation] \nBrazil \n```","['plaintext', 'summary', 'dynamic', 'software', 'engineer', 'year', 'progressive', 'experience', 'developing', 'robust', 'application', 'using', 'variety', 'programming', 'language', 'including', 'typescript', 'sql', 'java', 'proven', 'expertise', 'leveraging', 'web', 'database', 'technology', 'core', 'firebase', 'mongodb', 'highly', 'adept', 'employing', 'modern', 'framework', 'tool', 'streamline', 'development', 'process', 'enhance', 'application', 'performance', 'committed', 'delivering', 'innovative', 'solution', 'continuous', 'improvement', 'software', 'development', 'lifecycle', 'skill', 'program

In [9]:
def vectorize_text(tokens, w2v_model):
    # Vectorize the tokens (including bigrams)
    vectors = [w2v_model.wv[word] for word in tokens if word in w2v_model.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(w2v_model.vector_size)

In [5]:
# Create bigrams using Gensim's Phrases model
bigram_model = Phrases(job_postings_df['tokens'].tolist() + cvs_df['tokens'].tolist(), min_count=5, threshold=10)

In [6]:
# Apply the bigram model to your tokenized text
bigram_job_postings = [bigram_model[doc] for doc in job_postings_df['tokens']]
bigram_cvs = [bigram_model[doc] for doc in cvs_df['tokens']]

In [7]:
# Train Word2Vec model with CBOW (sg=0) and bigrams
w2v_model_bigrams = Word2Vec(sentences=bigram_job_postings, vector_size=100, window=5, min_count=2, sg=0, workers=4)

# Get the word vectors
word_vectors_bigrams = w2v_model_bigrams.wv

In [10]:
# Vectorize both job postings and CVs using the new bigram model
job_postings_df['vector'] = job_postings_df['tokens'].apply(lambda tokens: vectorize_text(bigram_model[tokens], w2v_model_bigrams))
cvs_df['vector'] = cvs_df['tokens'].apply(lambda tokens: vectorize_text(bigram_model[tokens], w2v_model_bigrams))

In [11]:
# Get the vectors
job_vectors = np.vstack(job_postings_df['vector'])
cv_vectors = np.vstack(cvs_df['vector'])

# Compute cosine similarity between CVs and job postings
similarities = cosine_similarity(cv_vectors, job_vectors)

# Get the top N most similar jobs for each CV
top_n = 10
top_indices = np.argsort(-similarities, axis=1)[:, :top_n]

In [12]:
all_recommendations = []
for cv_idx, job_idxs in tqdm(enumerate(top_indices)):
    for rank, job_idx in enumerate(job_idxs, start=1):  # Start rank from 1
        recs = job_postings_df.iloc[job_idx][['jobPostingId', 'title', 'description', 'gender_category']].copy()
        recs = recs.reset_index(drop=True) 
        recs['rank'] = rank 
        recs['cv_index'] = cv_idx           # Track which CV this is for
        recs['CV'] = cvs_df.iloc[cv_idx]['CV']
        recs['applicant_gender'] = cvs_df.iloc[cv_idx]['gender']
        all_recommendations.append(recs)

16000it [04:35, 58.06it/s]


In [13]:
# Create a DataFrame with the recommendations
recommendations_df = pd.DataFrame(all_recommendations)
recommendations_df = recommendations_df.rename(columns = {0: 'jobPostingId', 1:'title', 2:'description', 3:'gender_category'})
recommendations_df.head()

jobPostingId                                                     title  \
2335     3823301785                                    React Native Developer   
4053     3852604165  Sr. Associate, Software Engineer with Security Clearance   
24830    3879772473                                   Staff Software Engineer   
21739    3909968109                     Manager, Engineering - Cloud Security   
10482    3852303913                                        Frontend Developer   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                

In [14]:
# recommendations_df = pd.concat(all_recommendations, ignore_index=True)
cols = [ 'cv_index', 'CV', 'applicant_gender', 'jobPostingId', 'title', 'description', 'gender_category', 'rank']
recommendations_df = recommendations_df[cols] 
recommendations_df.head(10)

cv_index  \
2335          0   
4053          0   
24830         0   
21739         0   
10482         0   
24840         0   
4675          0   
36712         0   
11450         0   
2705          0   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     CV  \
2335   ```plaintext\n**Summary**  \nDynamic and detail-oriented software engineer with 12-14 years of progressive experience in developing robust applications using a variety of programming languages including TypeScript, SQL, and Java. Proven expertise in leveraging web and database technologies such as Node.js, ASP.NET Core, Firebase, and MongoDB. Highly adept in employing modern frameworks and tools to streamline development processes and enhance application performance. Committed to delivering innovative solutions and continuous improvement in the software development lifecycle.\n\n**Skills**  \n- **Programming Languages:** TypeScript, SQL, Java  \n- **Databases:** Firebase, MongoDB  \n- **Web Frameworks:** Node.js, ASP.NET Core  \n- **Other Frameworks:** .NET, Keras  \n- **Tools:** Git, Ansible  \n- **IDEs:** Sublime Text, IntelliJ  \n- **Operating Systems:** Windows, macOS  \n\n**Experience**  \n- Developed and maintained complex software applications, leading projects from conception through deployment.  \n- Collaborated with cross-functional teams to design and implement scalable solutions, enhancing user experience and operational efficiency.  \n- Utilized TypeScript and SQL for backend services and database management, demonstrating fluency in data-driven application development.  \n- Leveraged frameworks such as Node.js and ASP.NET Core to build responsive web applications, driving increased engagement and usability.  \n- Implemented CI/CD practices using Git and Ansible, resulting in improved deployment times and reduced rollout errors.  \n\n**Education**  \nBachelorâ€™s Degree in Computer Science  \n[Your University Name]  \n[Year of Graduation]  \nBrazil  \n```   
4053   ```plaintext\n**Summary**  \nDynamic and detail-oriented software engineer with 12-14 years of progressive experience in developing robust applications using a variety of programming languages including TypeScript, SQL, and Java. Proven expertise in leveraging web and database technologies such as Node.js, ASP.NET Core, Firebase, and MongoDB. Highly adept in employing moder

In [15]:
recommendations_df.to_csv(f'Recommendation Data/word2vec_cbow_bigrams_{len(recommendations_df)}.csv', index=False)